### After running the extrapolate script, you can use this script to check the results.

In [ ]:
import os
import json
from tqdm import tqdm
from collections import defaultdict

dir_name = 'results'
all_runnames = [r for r in os.listdir(dir_name) if os.path.isdir(os.path.join(dir_name, r))]
all_runnames.sort()

num_seeds = 8
all_runs = defaultdict(list)
for runname in tqdm(all_runnames):
    already_have_seeds = [int(f[4:-5]) for f in os.listdir(os.path.join(dir_name, runname)) if f.startswith('seed') and f.endswith('.json')]
    if len(already_have_seeds) < num_seeds:
        # get the missing seeds
        missing_seeds = set(range(num_seeds)) - set(already_have_seeds)
        print(f"Run {runname} is missing seeds: {missing_seeds}. Please run it again.")
        continue
    for seed in range(num_seeds):
        all_runs[runname].append(json.load(open(os.path.join(dir_name, runname, f'seed{seed}.json'))))

In [ ]:
import numpy as np

postfix = '-n4-t1.0-top_p0.7'
prev_type = ''
for runname, data_list in all_runs.items():
    cur_type = runname.split('-thresh')[0]
    if cur_type != prev_type:
        print(f"\n**{cur_type}**"); prev_type = cur_type
    method = 'thresh_' + runname.split('-thresh_')[1][:-len(postfix)]
    all_accs, all_lengths, all_resample_rates = [], [], []
    for ds in data_list:
        for dp in ds:
            all_accs.extend(dp['accs'])
            all_lengths.extend(dp['lengths'])
            all_resample_rates.extend(dp['resample_rates'])
    print(f'- `{method}`: acc={np.mean(all_accs):.4f}, resample rate={np.mean(all_resample_rates):.4f}, length={np.mean(all_lengths):.2f},')